<a href="https://colab.research.google.com/github/JaswanthJavangula/ML_Practise/blob/main/1ML%20JAshu/RandomForest%5BCLASS%26REGRE%5D/prac_RFC_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [87]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os

os.environ["KAGGLE_CACHE"] = "/content/sample_data"
# Set the path to the file you'd like to load
file_path = "Rewards.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "vysakhvms/rewards",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head(2))

/tmp/ipykernel_1243/3838998114.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'rewards' dataset.
First 5 records:    Gender  Age Membership Loyalty Points Discount Gift Earlybird    Reward
0    Male   45    Premium    501 to 1000      Yes  Yes       Yes  Cashback
1  Female   41     Normal      Below 500       No   No        No    Coupon


In [88]:
import pandas as pd
df = pd.DataFrame(df)
df.head(3)

,Gender,Age,Membership,Loyalty Points,Discount,Gift,Earlybird,Reward
0,Male,45,Premium,501 to 1000,Yes,Yes,Yes,Cashback
1,Female,41,Normal,Below 500,No,No,No,Coupon
2,Female,25,Premium,Above 1000,Yes,Yes,Yes,Coupon or Cashback


In [89]:
df['Loyalty Points'].value_counts()

,count
Loyalty Points,
501 to 1000,53
Above 1000,49
Below 500,48


In [90]:
df.isnull().sum()

,0
Gender,0
Age,0
Membership,0
Loyalty Points,0
Discount,0
Gift,0
Earlybird,0
Reward,0


In [91]:
df["Membership"].value_counts()

,count
Membership,
Normal,93
Premium,57


In [92]:
df['Gender'].value_counts()

,count
Gender,
Female,92
Male,58


In [93]:
df["Discount"].value_counts()

,count
Discount,
Yes,116
No,34


In [94]:
df['Gift'].value_counts()

,count
Gift,
No,79
Yes,71


In [95]:
df['Reward'].value_counts()

,count
Reward,
Coupon,79
Coupon or Cashback,49
Cashback,22


In [96]:
X = df.iloc[:,:-1]
y = df["Reward"].values
X.shape, y.shape

((150, 7), (150,))

In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Gender          150 non-null    object
 1   Age             150 non-null    int64 
 2   Membership      150 non-null    object
 3   Loyalty Points  150 non-null    object
 4   Discount        150 non-null    object
 5   Gift            150 non-null    object
 6   Earlybird       150 non-null    object
 7   Reward          150 non-null    object
dtypes: int64(1), object(7)
memory usage: 9.5+ KB


In [98]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify = y)

In [99]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
cat_features = list(X.select_dtypes(include = "object"))
num_features = list(X.select_dtypes(exclude = "object"))

oh = OneHotEncoder(drop = "first", handle_unknown="ignore")
sc = StandardScaler()

preprocessor = ColumnTransformer([
    ("One Hot", oh, cat_features),
    ("Standard scaler" , sc, num_features)
],remainder= "passthrough")

In [100]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [101]:
pd.DataFrame(X_train).head(3)

,0,1,2,3,4,5,6,7
0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.345559
1,0.0,1.0,1.0,0.0,1.0,1.0,1.0,-0.459627
2,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.479757


In [102]:
pd.DataFrame(X_test).head(3)

,0,1,2,3,4,5,6,7
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.687536
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.533209
2,0.0,1.0,1.0,0.0,1.0,1.0,1.0,-1.264813


In [103]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score, precision_score,roc_auc_score,accuracy_score,confusion_matrix, classification_report

In [104]:
# metrics funtion
def eval_metrics(true, predicted):
  accuracy = accuracy_score(true, predicted)
  precision = precision_score(true, predicted, average="macro")
  recall = recall_score(true, predicted, average="macro")
  f1 = f1_score(true, predicted, average="macro")
  return accuracy, precision, recall, f1

In [105]:
models = {
    "Log Regeression" : LogisticRegression(),
    "Decision Tree" : DecisionTreeClassifier(),
    "Random Forest" : RandomForestClassifier()
}

for i in range(len(list(models))):
  model = list(models.values())[i]
  model.fit(X_train, y_train)

  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  train_ACC , train_precision, train_recall , train_f1 = eval_metrics(y_train, y_train_pred)
  test_acc , test_precision, test_recall , test_f1 = eval_metrics(y_test, y_test_pred)

  print(list(models.keys())[i])
  print("\n - Model Performance in Training Set")
  print(f" - Accuracy : {train_ACC*100:.2f}%", f" ` Precision : {train_precision*100:.2f}%"),
  print(f" - Recall : {train_recall*100:.2f}%", f" ` F1 Score : {train_f1*100:.2f}%")
  print("\n - Model Performance in Testing Set")
  print(f" - Accuracy : {test_acc*100:.2f}%", f" ` Precision : {test_precision*100:.2f}%"),
  print(f" - Recall : {test_recall*100:.2f}%", f" ` F1 Score : {test_f1*100:.2f}%")
  print("="*35)
  print("\n")

Log Regeression

 - Model Performance in Training Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%

 - Model Performance in Testing Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%


Decision Tree

 - Model Performance in Training Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%

 - Model Performance in Testing Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%


Random Forest

 - Model Performance in Training Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%

 - Model Performance in Testing Set
 - Accuracy : 100.00%  ` Precision : 100.00%
 - Recall : 100.00%  ` F1 Score : 100.00%


